In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import os 
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# PLSSVD: validation on held-out trials

The primary analysis concerns **new trials from the same participants**. Participant subsamples and new matchings assess sensitivity, not generalisation to unseen participants. Since PLSSVD was chosen using previous results, this is conditional validation of that choice; an independent dataset would strengthen confirmation.

In [2]:
ROOT = Path.cwd()
if not (ROOT / 'utils_updated.py').exists() and (ROOT / 'iEEGvsMEG' / 'utils_updated.py').exists():
    ROOT = ROOT / 'iEEGvsMEG'
if not (ROOT / 'utils_updated.py').exists():
    raise FileNotFoundError('Set ROOT to the directory containing coverage_matching.ipynb and utils_updated.py.')
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))
sys.path.insert(0, str(ROOT.parent / 'LB'))
from src.setting import GetInfo, PROJECT_PATH
from plssvd_eval_utils import (
    ValidationOptions, prepare_trial_cache, load_trial_cache,
    validate_plssvd, plot_plssvd_validation,
)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

In [3]:
MEG_KIND = 'paired_coverage'  # full_average, full_concatenated, coverage_average, paired_coverage, random_control
options = ValidationOptions(repeats=5,                 # first: all participants; others: participant subsets + new matching/splits
                            candidates=(1, 2, 3, 5, 10),
                            n_null=199,                # increase for final analysis; minimum tail fraction = 1/(n_null+1)
                            seed=2026,
                            subject_fraction=0.8,
                            split_unit='trial',         # use 'group' with run/stimulus metadata when trials are dependent
                            block_scaling='none')
MEG_RAW_DIR = Path('/projects/MINDLAB2025_MEG-Auditory_Cognitive_Maps/scratch/APR2020_Block3_SingleTrial_BarbaraNikita')
TRIAL_METADATA_CSV = None       
CACHE_DIR = ROOT / 'out' / 'trial_cache'
OUTPUT_DIR = ROOT / 'out' / 'plssvd_eval' / MEG_KIND

## Participants and coordinates
Same discovery and coordinate setup as the previous notebooks. The averaged MEG filenames identify participants only; model validation loads their original trials. Verify the inherited coordinate conversion against your original coordinate units.

In [4]:
MEG_DIR = ROOT / 'MEG' / 'dataMEG'
IEEG_DIR = ROOT / 'ieeg_shortWOBS_fs250'

meg_subj_list = sorted(file.removesuffix('_source.p') for file in os.listdir(MEG_DIR) if file.endswith('_source.p'))
print('MEG Subject number : ', len(meg_subj_list))

ieeg_subj_list = sorted(file.removesuffix('_epochs.p') for file in os.listdir(IEEG_DIR) if file.endswith('_epochs.p'))
if not meg_subj_list or not ieeg_subj_list:
    raise FileNotFoundError('Check MEG_DIR and IEEG_DIR: no source or epoch files found.')
print('iEEG Subject number : ', len(ieeg_subj_list))

info_file = IEEG_DIR / f'{ieeg_subj_list[0]}_info.json'
with open(info_file) as json_data:
    d = json.load(json_data)
    time = d['time_epoch']

SFREQ = 250
CONDITIONS = (1, 2)    

project_path = ROOT.parent / PROJECT_PATH
coord, areas, elect_list, subj_list, regions_ieeg = GetInfo(ieeg_subj_list, data_path=str(IEEG_DIR), project_path=str(project_path))
coord=np.array(coord)
coord = np.where(abs(coord) >100, coord/1000, coord)
coord = np.where(abs(coord) >100, coord/1000, coord)
coord = coord/1000

MEG Subject number :  32
iEEG Subject number :  30


In [5]:
electrode_metadata = pd.DataFrame(coord, columns=['x', 'y', 'z'])
electrode_metadata['subject'] = subj_list
electrode_metadata['channel'] = elect_list
electrode_metadata['region'] = regions_ieeg
electrode_metadata['channel_index'] = electrode_metadata.groupby('subject', sort=False).cumcount()

## Load unaveraged trials
The original MEG `*_source.p` files contain trial averages and cannot support this validation. The exporter uses the `OUT.sources_ERFs` layout from `LB10_extract_data.py` and requires `mat73` when exporting MAT files. iEEG uses the existing epochs and event labels. Cache files are memory-mapped; export requires disk space for all selected trials. Actual MEG and iEEG time vectors must agree; an extra final MEG sample is removed only after checking its time vector.

Optional metadata CSV columns:

| Column | Meaning |
|---|---|
| modality | `ieeg` or `meg` |
| subject | Filename participant identifier |
| condition | Original condition code, e.g. 1 or 2 |
| trial_index | Zero-based trial index **within that condition** |
| split_group | Run or repeated stimulus identity; with `split_unit='group'`, never crosses partitions |
| permutation_block | Acoustic or other exchangeability stratum for condition-label shuffling |

When supplied, the CSV must cover every selected trial. Permutations also respect `split_group` when present. Use group splitting for dependent trials, including repeated stimuli where appropriate. Each condition needs at least eight trials, with at least two in each partition; group splitting may need more. A cache made with different metadata/configuration must be rebuilt in a new directory.

All learned normalisation **in this notebook** is fitted on training data. This cannot undo leakage from upstream across-trial normalisation or data-derived source filters; those must be independent or fitted within folds for a fully strict evaluation.

In [6]:
meg_subj_list.remove('SUBJ_0038') # does not have enough trials

In [ ]:
prepare_trial_cache(
    MEG_RAW_DIR, IEEG_DIR, CACHE_DIR,
    meg_subj_list, ieeg_subj_list, electrode_metadata,
    conditions=CONDITIONS, ieeg_coordinate_unit='m', meg_coordinate_unit='m',
    trial_metadata_csv=TRIAL_METADATA_CSV,
)

In [ ]:
trials = load_trial_cache(CACHE_DIR)
display(pd.DataFrame([{'modality': modality, 'subject': subject.subject, 'condition': condition,'n_trials': len(array), 'n_channels': array.shape[1], 'n_times': array.shape[2]} 
                      for modality in ('ieeg', 'meg') for subject in getattr(trials, modality) for condition, array in zip(trials.conditions, subject.data)]))

## Fit, select, and test
Trials are split independently within each modality and participant: approximately 50% train, 20% tune, and two 15% test halves, adjusted to retain at least two trials per condition per partition. No neighbouring time samples are randomly split. Both conditions are stacked along the observation axis to preserve condition differences.

MEG source normalisation is estimated from training condition averages; iEEG retains the fixed ×1000 conversion. Coverage matching is fixed across partitions within a repetition. PLSSVD uses training data only. Component count maximises mean bidirectional **tuning Q² for the original signals**, with fixed training ridge prediction maps; smaller counts win ties. This compares the same target feature space for every count. Test component order and signs remain fixed from training.

Repetition 0 is the predeclared primary full-cohort analysis. Later repetitions change participant subsets, trial splits, and applicable matching realisations. They are overlapping sensitivity analyses, not independent replicates or confidence intervals. Component identities may change across fits; interpret their aggregate metrics, not component-by-component trajectories across repetitions.

In [ ]:
result = validate_plssvd(trials, MEG_KIND, options, output_dir=OUTPUT_DIR)
display(result['summary'].round(3))
display(result['selection'].query('repeat == 0').round(3))

NameError: name 'trials' is not defined

## Held-out correspondence and reliability

- **test_r:** signed Pearson correlation of corresponding fixed PLS scores across condition × time. High held-out correlation supports reproducible temporal correspondence; it can still be driven by tone responses.
- **Q²:** 1 − prediction error / error of predicting the training feature mean. Positive values improve on this baseline; negative values are worse. This is prediction of independently averaged signals, not individual paired trials.
- **contrast_r:** correlation of condition 2 minus condition 1 time courses. Signed values show direction; the aggregate absolute statistic tests association regardless of direction. Inspect contrast RMS too: correlation alone does not establish a substantial modulation.
- **split_half_r:** agreement between independent test-half scores with fixed training weights. Contrast reliability is reported separately.
- **pattern_split_half_r:** agreement of forward regression patterns estimated separately in each test half. These quantify spatial reproducibility conditional on the training axes, not stability of independently refitted PLS weights.

Reliability correlations are raw split-half estimates, without a Spearman–Brown correction or a calibrated noise ceiling. Poor reliability limits interpretation of low cross-modal agreement.

In [ ]:
display(result['components'].query('repeat == 0').round(3))
plot_plssvd_validation(result)

In [ ]:
primary = result['components'].query('repeat == 0')
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for modality in ('ieeg', 'meg'):
    axes[0].plot(primary.component, primary[f'{modality}_pattern_split_half_r'], 'o-', label=modality)
    axes[1].plot(primary.component, primary[f'{modality}_contrast_split_half_r'], 'o-', label=modality)
for ax, title in zip(axes, ('Forward-pattern reliability', 'Condition-contrast reliability')):
    ax.set(title=title, xlabel='Component', ylabel='Test-half Pearson r', ylim=(-1, 1))
    ax.legend()
plt.show()

## Null models: distinct questions
Nulls are evaluated only for the primary fit, keeping preprocessing, selected component count, and weights fixed. Statistics aggregate the selected components; no test-driven sign flips or component matching are applied.

1. **Temporal shift:** shifts all MEG test scores by the same circular offset within each condition. Tests sensitivity to temporal alignment while preserving circular temporal structure. Epoch-locked responses are nonstationary, so this tail fraction is a **surrogate diagnostic**, not automatically a valid permutation p-value.
2. **Condition labels:** permutes MEG held-out trial labels within participant and exchangeability blocks, preserving counts, then reconstructs projected condition contrasts using the same aggregation. A plus-one permutation tail is interpretable conditionally on exchangeable labels. Blocks containing only one condition cannot be shuffled. Acoustic confounding is not removed by simply labelling this a memory test.
3. **Channel correspondence:** permutes held-out MEG forward-pattern rows within iEEG participant and region (when available). Tests the specified spatial correspondence conditional on the fitted axes. Spatial autocorrelation can violate row exchangeability: treat this as a diagnostic unless those assumptions are justified.

Do not pool tail fractions across sensitivity repetitions. Report all planned tests and account for multiple inferential claims. The mean tone scaffold cancels in the condition difference only to the extent that it is shared across conditions. A recognition-memory claim still requires known condition semantics and adequate control of acoustic/sequence differences.

In [ ]:
display(result['null_tests'])
print(f'Outputs saved to: {OUTPUT_DIR}')

## Saved results and interpretation
The output folder contains summary/component/tuning tables, trial partition audits, participant selections, matching and feature metadata, trained weights and means, preprocessing parameters, prediction maps, projected scores, and primary null distributions. Use a different `OUTPUT_DIR` to retain separate configurations.

Evidence is strongest when held-out correspondence and prediction are positive, condition contrasts are internally reliable and cross-modally related under justified label permutations, and conclusions survive participant/matching sensitivity analyses. These results alone do not establish unseen-participant generalisation, identical spatial organisation, or a memory-specific mechanism.